In [22]:
api_base = r'http://footballapi.eastus.azurecontainer.io:3000/'

In [23]:
import requests
import pandas as pd
import json
from pandas import DataFrame

# getting the id for the last n matches

In [24]:


def get_team_lnm(api_base :str , team_id:int , num_matchs: int) -> dict :
  '''this function takes the base of the api without the endpoint , the team id and the number of last matchs wanted
  and returns a dictionary of the last matchs ids of the team with the home team and away team names'''
  try :
    response = requests.get(api_base +f'teams/{team_id}/events/last/0') # api response
  except:
    return 'the api is down'
  match_info = { # extracting match details
    match['id']: {
        'homeTeam': match['homeTeam']['name'],
        'awayTeam': match['awayTeam']['name']
    }
    for match in response.json()['events']
}
  match_info =  dict(reversed(list(match_info.items())[-num_matchs:])) # filtering the last n matches from the data and reverse them (the last match is the first)
  match_info['target_team_id'] =team_id  # adding the id of the team to the dict
  match_info['target_team_name'] = requests.get(api_base + f'teams/{team_id}').json( ).get('team').get('name') # adding the name of the team to the dict
  return match_info

teams_info = get_team_lnm(api_base , 2829 , 5)
print(teams_info )

{14083112: {'homeTeam': 'Rayo Vallecano', 'awayTeam': 'Real Madrid'}, 14566636: {'homeTeam': 'Liverpool', 'awayTeam': 'Real Madrid'}, 14083099: {'homeTeam': 'Real Madrid', 'awayTeam': 'Valencia'}, 14083729: {'homeTeam': 'Real Madrid', 'awayTeam': 'Barcelona'}, 14566596: {'homeTeam': 'Real Madrid', 'awayTeam': 'Juventus'}, 'target_team_id': 2829, 'target_team_name': 'Real Madrid'}


# getting the statistics for the last n matches using id

In [29]:

def get_match_stats(api_base: str , matches_info: dict ) -> DataFrame :
    '''this function takes the base of the api without the endpoint , the match onfo dictionary  and returns the statistics of the matches as a dataframe'''
    all_matches_stats = [] # define a dict to contain all the information to convert it to a DataFrame later
    
    for match_id in matches_info.keys():
        if isinstance(match_id, int):
            # getting the correct key for the values based on whether the team is away or home
            if matches_info.get('target_team_name') == matches_info.get(match_id).get('awayTeam'):
                value_key = 'awayValue'
            else :
                value_key = 'homeValue'
                
            match_stats = {}
            home_team = matches_info[match_id]['homeTeam'] # just adding each match id , home team name and away team name to the dict
            away_team = matches_info[match_id]['awayTeam']
            match_stats['match_id'] = match_id
            match_stats['home_team'] = home_team
            match_stats['away_team'] = away_team
            
            try : # handling the api if it's disconnected or failed
                response = requests.get(api_base + f'events/{match_id}/statistics').json()['statistics'] # getting the statistics of the match
            except :
                return 'api is down'
            
            for period in response: # loop over each period statistics
                if period.get('period') ==  'ALL': # get the general stats only
                    response = period['groups'] # filtering only the overall statistics
                
            for group_stat in response: # loop over each match group statistics
                for stat_item in group_stat['statisticsItems']: # loop over each statistic item (collection of stats under a specific category)
                    if stat_item.get('name') != None :
                        match_stats[stat_item.get('name')] = stat_item.get(value_key) # adding the feature and the value for it in the dict
            
            all_matches_stats.append(match_stats) # adding the whole match stats to the general list

    all_matches_stats = pd.DataFrame(all_matches_stats ).fillna(0) # convert the general list to data frame and fill none values with 0
    return all_matches_stats
        
        
        
matches_stats = get_match_stats(api_base ,teams_info )
matches_stats

,match_id,home_team,away_team,Ball possession,Expected goals,Big chances,Total shots,Goalkeeper saves,Corner kicks,Fouls,...,Goal kicks,Distance covered,Number of sprints,Big chances scored,Errors lead to a shot,Penalty saves,Red cards,Through balls,Errors lead to a goal,Punches
0,14083112,Rayo Vallecano,Real Madrid,54,0.98,1,21,2,8,7,...,17,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,14566636,Liverpool,Real Madrid,61,0.45,1,8,8,2,11,...,4,112.72776,152.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,14083099,Real Madrid,Valencia,65,2.71,4,21,1,7,14,...,2,0.00000,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
3,14083729,Real Madrid,Barcelona,32,3.63,4,23,4,12,12,...,6,0.00000,0.0,2.0,1.0,0.0,1.0,2.0,1.0,1.0
4,14566596,Real Madrid,Juventus,66,2.69,3,28,4,13,10,...,4,105.02857,124.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0


# getting the statistics for every player in the match (in match statistics)

In [ ]:
def get_players_stats(api_base: str , matches_info: dict ) -> DataFrame :
    pass

# getting the real position for the player in the field

# calculating the score for each player

# getting the most recommended players for the match 

# determining the best plan to play with and suggestion